In [ ]:
#Sistema
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#NLP
import spacy
from transformers import pipeline
from tqdm import tqdm

#Scikit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    roc_auc_score,
)

#Deep Learning (TensorFlow / Keras)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Utils
import joblib



In [ ]:

RANDOM_STATE = 42
SAMPLE_PER_CLASS = 80000

# spaCy: se der erro de modelo não encontrado, rode no terminal:
# python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


In [4]:
# %%
# ==============================
# 1) Leitura + filtro narrativas
# ==============================
df = pd.read_csv("complaints.csv")
df_backup = df.copy()


In [5]:
df.shape

(13682539, 18)

In [6]:

df = df[df["Consumer complaint narrative"].notnull()].copy()

In [7]:
df.shape

(3717193, 18)

In [8]:
def consolidar_produtos(df_in: pd.DataFrame) -> pd.DataFrame:
    mapping = {
        # credit reporting
        "Credit reporting": "credit reporting",
        "Credit reporting or other personal consumer reports": "credit reporting",
        "Credit reporting, credit repair services, or other personal consumer reports": "credit reporting",

        # debt collection
        "Debt collection": "debt collection",
        "Debt or credit management": "debt collection",

        # mortgages and loans
        "Mortgage": "mortgages and loans",
        "Vehicle loan or lease": "mortgages and loans",
        "Consumer Loan": "mortgages and loans",
        "Student loan": "mortgages and loans",
        "Payday loan": "mortgages and loans",
        "Payday loan, title loan, or personal loan": "mortgages and loans",
        "Payday loan, title loan, personal loan, or advance loan": "mortgages and loans",

        # credit cards
        "Credit card": "credit cards",
        "Credit card or prepaid card": "credit cards",
        "Prepaid card": "credit cards",

        # retail banking
        "Checking or savings account": "retail banking",
        "Bank account or service": "retail banking",
        "Money transfers": "retail banking",
        "Money transfer, virtual currency, or money service": "retail banking",
        "Virtual currency": "retail banking",
        "Other financial service": "retail banking",
    }

    df_out = df_in.copy()
    df_out["Product Consolidated"] = df_out["Product"].map(mapping)
    return df_out


df = consolidar_produtos(df)
df = df[df["Product Consolidated"].notnull()].copy()

print("Distribuição (Product Consolidated):")
print(df["Product Consolidated"].value_counts())


Distribuição (Product Consolidated):
Product Consolidated
credit reporting       2497507
debt collection         408350
retail banking          295820
mortgages and loans     288388
credit cards            227128
Name: count, dtype: int64


In [9]:
df_sample = (
    df.groupby("Product Consolidated", group_keys=False)
      .apply(lambda x: x.sample(n=min(len(x), SAMPLE_PER_CLASS), random_state=RANDOM_STATE))
      .reset_index(drop=True)
)

print("\nAmostra balanceada (80k por classe):")
print(df_sample["Product Consolidated"].value_counts())
print("df_sample shape:", df_sample.shape)


Amostra balanceada (80k por classe):
Product Consolidated
credit cards           80000
credit reporting       80000
debt collection        80000
mortgages and loans    80000
retail banking         80000
Name: count, dtype: int64
df_sample shape: (400000, 19)


In [10]:
negative_keywords = [
    "fraud", "unauthorized", "error", "wrong", "charged",
    "denied", "complaint", "issue", "problem",
    "not resolved", "never received", "scam",
    "harassment", "threat", "foreclosure", "late fee",
    "debt", "collection", "dispute", "inaccurate",
    "overcharge", "misleading"
]
positive_keywords = [
    "resolved", "thank you", "satisfied",
    "helpful", "fixed", "appreciate",
    "closed my complaint", "solved"
]

def create_sentiment_rule(text: str):
    text = str(text).lower()
    neg_count = sum(k in text for k in negative_keywords)
    pos_count = sum(k in text for k in positive_keywords)

    if neg_count > pos_count:
        return 0
    elif pos_count > neg_count:
        return 1
    return np.nan

df_sample["sentiment_rule"] = df_sample["Consumer complaint narrative"].apply(create_sentiment_rule)
df_rule = df_sample[df_sample["sentiment_rule"].notnull()].copy()

print("\nDistribuição do target (sentiment_rule):")
print(df_rule["sentiment_rule"].value_counts(normalize=True))



Distribuição do target (sentiment_rule):
sentiment_rule
0.0    0.960316
1.0    0.039684
Name: proportion, dtype: float64


In [11]:

df_compare = df_rule.sample(15000, random_state=RANDOM_STATE).copy()

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis",
    device=0  # se der erro, use -1
)

batch_size = 32
texts = df_compare["Consumer complaint narrative"].tolist()

predictions = []
for i in tqdm(range(0, len(texts), batch_size), desc="HF sentiment"):
    batch = texts[i:i + batch_size]
    outputs = sentiment_pipeline(batch, truncation=True, max_length=512)
    predictions.extend(outputs)

df_compare["hf_label"] = [p["label"] for p in predictions]
df_compare["hf_score"] = [p["score"] for p in predictions]

df_compare["sentiment_hf"] = df_compare["hf_label"].map({
    "positive": 1,
    "negative": 0,
    "neutral": np.nan
})

df_compare = df_compare[df_compare["sentiment_hf"].notnull()].copy()

print("\nComparação Heurística vs Transformer (binário, sem neutral):")
print("Distribuição Heurística (rule):")
print(df_compare["sentiment_rule"].value_counts(normalize=True))
print("\nDistribuição Transformer (hf):")
print(df_compare["sentiment_hf"].value_counts(normalize=True))
print("\nMatriz de Confusão (rule como referência):")
print(confusion_matrix(df_compare["sentiment_rule"], df_compare["sentiment_hf"]))
print("\nClassification report (rule como referência):")
print(classification_report(df_compare["sentiment_rule"], df_compare["sentiment_hf"]))


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 702.60it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
HF sentiment: 100%|██████████| 469/469 [15:24<00:00,  1.97s/it]


Comparação Heurística vs Transformer (binário, sem neutral):
Distribuição Heurística (rule):
sentiment_rule
0.0    0.971443
1.0    0.028557
Name: proportion, dtype: float64

Distribuição Transformer (hf):
sentiment_hf
0.0    0.957544
1.0    0.042456
Name: proportion, dtype: float64

Matriz de Confusão (rule como referência):
[[3695  149]
 [  94   19]]

Classification report (rule como referência):
              precision    recall  f1-score   support

         0.0       0.98      0.96      0.97      3844
         1.0       0.11      0.17      0.14       113

    accuracy                           0.94      3957
   macro avg       0.54      0.56      0.55      3957
weighted avg       0.95      0.94      0.94      3957



In [12]:
def create_sentiment_final(response: str) -> int:
    if response in ["Closed with monetary relief", "Closed with non-monetary relief"]:
        return 1
    return 0

df_sample["sentiment_final"] = df_sample["Company response to consumer"].apply(create_sentiment_final)

print("\nDistribuição Company response to consumer (na amostra):")
print(df_sample["Company response to consumer"].value_counts())
print("\nDistribuição do target final (sentiment_final):")
print(df_sample["sentiment_final"].value_counts(normalize=True))



Distribuição Company response to consumer (na amostra):
Company response to consumer
Closed with explanation            304426
Closed with non-monetary relief     65795
Closed with monetary relief         26872
Untimely response                    1931
Closed                                859
In progress                           117
Name: count, dtype: int64

Distribuição do target final (sentiment_final):
sentiment_final
0    0.768332
1    0.231667
Name: proportion, dtype: float64


In [13]:
def normalize_for_spacy(text: str) -> str:
    text = str(text).lower()
    # remove tokens só de x (xx, xxx, xxxx etc.)
    text = re.sub(r"\b[x]{2,}\b", " ", text)
    # remove números isolados
    text = re.sub(r"\b\d+\b", " ", text)
    # normaliza espaços
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_sample["text_norm"] = df_sample["Consumer complaint narrative"].apply(normalize_for_spacy)

processed = []
for doc in tqdm(nlp.pipe(df_sample["text_norm"].tolist(), batch_size=1000), total=len(df_sample), desc="spaCy lemmatization"):
    tokens = [
        t.lemma_
        for t in doc
        if t.is_alpha
        and not t.is_stop
        and not t.is_punct
        and len(t) > 2
    ]
    processed.append(" ".join(tokens))

df_sample["text_processed"] = processed


spaCy lemmatization: 100%|██████████| 400000/400000 [2:11:57<00:00, 50.52it/s]   


In [14]:
df_sample.to_csv("df_sample_processed.csv", index=False)

In [15]:
df_sample.to_parquet("df_sample_processed.parquet", index=False)

In [16]:
df_sample[["Consumer complaint narrative","text_processed"]].head(3)

,Consumer complaint narrative,text_processed
0,Review the attached documents. I ask the burea...,review attach document ask bureau commence inq...
1,I had paid off this card within one year of my...,pay card year card use avoid interest statemen...
2,This is my second complaint against this compa...,second complaint company open issue second acc...


In [25]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words=None,
    ngram_range=(1, 2),
    min_df=5,
    token_pattern=r"\b[a-zA-Z]{3,}\b"
)

X = vectorizer.fit_transform(df_sample["text_processed"])
y = df_sample["sentiment_final"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Logistic Regression (baseline forte para TF-IDF)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n=== Logistic Regression ===")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

try:
    y_proba = model.predict_proba(X_test)[:, 1]
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
except Exception as e:
    print("ROC-AUC não calculado:", e)

# Random Forest (comparação; pode ser lento em TF-IDF)
rf = RandomForestClassifier(
    n_estimators=50,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest (50 trees) ===")
print(classification_report(y_test, y_pred_rf))



=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.80      0.94      0.87     61467
           1       0.56      0.23      0.33     18533

    accuracy                           0.78     80000
   macro avg       0.68      0.59      0.60     80000
weighted avg       0.75      0.78      0.74     80000

Accuracy: 0.7790625
ROC-AUC: 0.763832204276337

=== Random Forest (50 trees) ===
              precision    recall  f1-score   support

           0       0.80      0.95      0.87     61467
           1       0.57      0.21      0.31     18533

    accuracy                           0.78     80000
   macro avg       0.69      0.58      0.59     80000
weighted avg       0.75      0.78      0.74     80000



In [18]:

feature_names = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

top_positive = coefs.argsort()[-20:]
top_negative = coefs.argsort()[:20]

print("\nTop termos associados a POSITIVO (relief):")
print([feature_names[i] for i in top_positive])

print("\nTop termos associados a NEGATIVO (sem relief):")
print([feature_names[i] for i in top_negative])



Top termos associados a POSITIVO (relief):
['direct express', 'bluebird', 'bank america', 'penfe', 'rush', 'midland', 'credit management', 'spring', 'comenity', 'bpo', 'information service', 'consultant', 'mrs', 'mcm', 'elan', 'portfolio recovery', 'resource management', 'transunion', 'credence resource', 'credence']

Top termos associados a NEGATIVO (sem relief):
['cash app', 'cashapp', 'zelle', 'nelnet', 'vanilla', 'cooper', 'netspend', 'robinhood', 'loancare', 'ocwen', 'inc', 'fcra experian', 'transworld', 'hsbc', 'nationstar', 'westlake', 'edfinancial', 'freedom mortgage', 'aidvantage', 'avant']


In [19]:

df_neg = df_sample[df_sample["sentiment_final"] == 0].copy()

# Reaproveita a coluna processada (limpa + lematizada)
# Se quiser, poderia usar df_neg["text_processed"] diretamente (já existe via df_sample)
vectorizer_neg = TfidfVectorizer(
    max_features=2000,
    stop_words=None,
    ngram_range=(1, 2),
    min_df=10,
    token_pattern=r"\b[a-zA-Z]{3,}\b"
)

results = []

for product in df_neg["Product Consolidated"].unique():
    df_temp = df_neg[df_neg["Product Consolidated"] == product]
    X_temp = vectorizer_neg.fit_transform(df_temp["text_processed"])

    mean_tfidf = X_temp.mean(axis=0).A1
    feature_names_temp = vectorizer_neg.get_feature_names_out()

    top_indices = mean_tfidf.argsort()[-15:]
    terms = [feature_names_temp[i] for i in top_indices]

    print(f"\n====== {product.upper()} ======")
    print(terms)

    for idx in top_indices:
        results.append({
            "Product": product,
            "Term": feature_names_temp[idx],
            "Score": float(mean_tfidf[idx])
        })

df_terms = (
    pd.DataFrame(results)
      .sort_values(["Product", "Score"], ascending=[True, False])
      .reset_index(drop=True)
)

print("\nTabela final (df_terms) pronta para relatório:")
print(df_terms.head(20))



====== CREDIT CARDS ======
['balance', 'dispute', 'call', 'tell', 'time', 'receive', 'report', 'pay', 'bank', 'credit card', 'charge', 'payment', 'account', 'credit', 'card']

====== CREDIT REPORTING ======
['inaccurate', 'item', 'request', 'section', 'remove', 'dispute', 'payment', 'inquiry', 'reporting', 'consumer', 'credit report', 'information', 'account', 'report', 'credit']

====== DEBT COLLECTION ======
['owe', 'letter', 'request', 'send', 'receive', 'credit report', 'pay', 'information', 'call', 'company', 'collection', 'report', 'account', 'credit', 'debt']

====== MORTGAGES AND LOANS ======
['bank', 'report', 'send', 'month', 'call', 'company', 'receive', 'time', 'tell', 'credit', 'account', 'mortgage', 'pay', 'loan', 'payment']

====== RETAIL BANKING ======
['zelle', 'receive', 'transfer', 'deposit', 'send', 'cash app', 'tell', 'app', 'transaction', 'cash', 'check', 'fund', 'money', 'bank', 'account']

Tabela final (df_terms) pronta para relatório:
             Product     

In [ ]:
# ==========================================================
# TF-IDF (5000) + TRAIN/TEST + DEEP LEARNING (Keras MLP)
# ==========================================================


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


# --------------------------
# 3) Sparse -> Dense (para Keras)
# --------------------------
X_train_dense = X_train.toarray().astype("float32")
X_test_dense  = X_test.toarray().astype("float32")

y_train_np = y_train.to_numpy(dtype="int32")
y_test_np  = y_test.to_numpy(dtype="int32")

# --------------------------
# 4) Modelo Deep Learning (MLP)
# --------------------------
input_dim = X_train_dense.shape[1]

model_dl = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

model_dl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.AUC(name="auc")
    ]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=2,
        restore_best_weights=True
    )
]

history = model_dl.fit(
    X_train_dense, y_train_np,
    validation_split=0.2,
    epochs=10,
    batch_size=512,
    callbacks=callbacks,
    verbose=1
)

# --------------------------
# 5) Avaliação
# --------------------------
y_proba_dl = model_dl.predict(X_test_dense, batch_size=1024).ravel()
y_pred_dl = (y_proba_dl >= 0.5).astype(int)

print("\n=== Deep Learning (Keras MLP) ===")
print(classification_report(y_test_np, y_pred_dl))
print("Accuracy:", accuracy_score(y_test_np, y_pred_dl))
print("ROC-AUC:", roc_auc_score(y_test_np, y_proba_dl))

# --------------------------
# 6) Salvar (opcional)
# --------------------------
# model_dl.save("models/keras_mlp_tfidf_5000.keras")

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.7746 - auc: 0.7452 - loss: 0.4708 - val_accuracy: 0.7767 - val_auc: 0.7662 - val_loss: 0.4570
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.7862 - auc: 0.7842 - loss: 0.4448 - val_accuracy: 0.7806 - val_auc: 0.7710 - val_loss: 0.4546
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.8003 - auc: 0.8143 - loss: 0.4209 - val_accuracy: 0.7776 - val_auc: 0.7681 - val_loss: 0.4631
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.8208 - auc: 0.8536 - loss: 0.3828 - val_accuracy: 0.7720 - val_auc: 0.7573 - val_loss: 0.4854
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step

=== Deep Learning (Keras MLP) ===
              precision    recall  f1-score   support

           0       0.81      0.93      0.87     61467
           1       0.56      0.29      0.38     18533

    accuracy                           0.78     80000
   macro avg       0.69      0.61      0.62     80000
weighted a

In [28]:
import os

os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)

# salvar métricas DL
with open("outputs/metrics_deep_learning.txt", "w") as f:
    f.write("=== Deep Learning (Keras MLP) ===\n\n")
    f.write("Treino (última época exibida):\n")
    f.write("Epoch 1: acc 0.7746 | auc 0.7452 | val_auc 0.7662\n")
    f.write("Epoch 2: acc 0.7862 | auc 0.7842 | val_auc 0.7710\n")
    f.write("Epoch 3: acc 0.8003 | auc 0.8143 | val_auc 0.7681\n")
    f.write("Epoch 4: acc 0.8208 | auc 0.8536 | val_auc 0.7573\n\n")

    f.write("Resultados (teste):\n")
    f.write(classification_report(y_test_np, y_pred_dl))
    f.write(f"\nAccuracy: {accuracy_score(y_test_np, y_pred_dl)}\n")
    f.write(f"ROC-AUC: {roc_auc_score(y_test_np, y_proba_dl)}\n")

# salvar o modelo DL
model_dl.save("models/keras_mlp_tfidf_5000.keras")
print("Salvo: outputs/metrics_deep_learning.txt e models/keras_mlp_tfidf_5000.keras")

Salvo: outputs/metrics_deep_learning.txt e models/keras_mlp_tfidf_5000.keras


In [22]:

OUTPUT_DIR = "outputs"
MODEL_DIR = "models"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("\nSalvando artefatos do pipeline...")

# ==========================================================
# 1️⃣ Dataset processado
# ==========================================================

print("Salvando dataset processado...")

df_sample.to_parquet(
    os.path.join(OUTPUT_DIR, "df_sample_processed.parquet"),
    index=False
)

# ==========================================================
# 2️⃣ Vectorizer
# ==========================================================

print("Salvando TF-IDF vectorizer...")

joblib.dump(
    vectorizer,
    os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl")
)

# ==========================================================
# 3️⃣ Modelos treinados
# ==========================================================

print("Salvando modelos treinados...")

joblib.dump(
    model,
    os.path.join(MODEL_DIR, "logistic_regression_model.pkl")
)

joblib.dump(
    rf,
    os.path.join(MODEL_DIR, "random_forest_model.pkl")
)

# ==========================================================
# 4️⃣ Métricas Logistic Regression
# ==========================================================

print("Salvando métricas Logistic Regression...")

with open(os.path.join(OUTPUT_DIR, "metrics_logistic_regression.txt"), "w") as f:

    f.write("=== Logistic Regression ===\n\n")

    f.write(classification_report(y_test, y_pred))

    f.write("\nAccuracy: " + str(accuracy_score(y_test, y_pred)) + "\n")

    try:
        y_proba = model.predict_proba(X_test)[:, 1]
        f.write("ROC-AUC: " + str(roc_auc_score(y_test, y_proba)) + "\n")
    except:
        f.write("ROC-AUC não calculado\n")

# ==========================================================
# 5️⃣ Métricas Random Forest
# ==========================================================

print("Salvando métricas Random Forest...")

with open(os.path.join(OUTPUT_DIR, "metrics_random_forest.txt"), "w") as f:

    f.write("=== Random Forest ===\n\n")

    f.write(classification_report(y_test, y_pred_rf))

# ==========================================================
# 6️⃣ Termos mais importantes do modelo
# ==========================================================

print("Salvando termos mais importantes...")

feature_names = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

top_positive = coefs.argsort()[-20:]
top_negative = coefs.argsort()[:20]

df_terms_model = pd.DataFrame({
    "positive_terms": [feature_names[i] for i in top_positive],
    "negative_terms": [feature_names[i] for i in top_negative]
})

df_terms_model.to_csv(
    os.path.join(OUTPUT_DIR, "model_terms.csv"),
    index=False
)

# ==========================================================
# 7️⃣ Previsões de teste
# ==========================================================

print("Salvando previsões do teste...")

df_predictions = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred_logreg": y_pred,
    "y_pred_rf": y_pred_rf
})

df_predictions.to_csv(
    os.path.join(OUTPUT_DIR, "model_predictions.csv"),
    index=False
)

# ==========================================================
# 8️⃣ Dores por produto (se existir df_terms)
# ==========================================================

if "df_terms" in globals():

    print("Salvando dores por produto...")

    df_terms.to_csv(
        os.path.join(OUTPUT_DIR, "pain_terms_by_product.csv"),
        index=False
    )

print("\nPipeline finalizado e artefatos salvos.")


Salvando artefatos do pipeline...
Salvando dataset processado...
Salvando TF-IDF vectorizer...
Salvando modelos treinados...
Salvando métricas Logistic Regression...
Salvando métricas Random Forest...
Salvando termos mais importantes...
Salvando previsões do teste...
Salvando dores por produto...

Pipeline finalizado e artefatos salvos.


In [ ]:


# =========================================
# CONFIG
# =========================================
OUTPUT_DIR = "outputs"
TOP_N_TERMS = 15

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================
# 1) Filtrar negativos (dores)
# =========================================
df_neg = df_sample[df_sample["sentiment_final"] == 0].copy()

# Se quiser usar "Consumer complaint narrative" no lugar:
# TEXT_COL = "Consumer complaint narrative"
TEXT_COL = "text_processed"

# =========================================
# 2) Extrair top termos por produto via TF-IDF médio
# =========================================
vectorizer_neg = TfidfVectorizer(
    max_features=2000,
    stop_words=None,  # já removido no text_processed
    ngram_range=(1, 2),
    min_df=10,
    token_pattern=r"\b[a-zA-Z]{3,}\b"
)

results = []

for product in sorted(df_neg["Product Consolidated"].dropna().unique()):
    df_temp = df_neg[df_neg["Product Consolidated"] == product]
    texts = df_temp[TEXT_COL].astype(str)

    X_temp = vectorizer_neg.fit_transform(texts)
    mean_tfidf = X_temp.mean(axis=0).A1
    feature_names = vectorizer_neg.get_feature_names_out()

    top_idx = mean_tfidf.argsort()[-TOP_N_TERMS:][::-1]  # desc
    for idx in top_idx:
        results.append({
            "Product": product,
            "Term": feature_names[idx],
            "Score": float(mean_tfidf[idx])
        })

df_terms = pd.DataFrame(results).sort_values(["Product", "Score"], ascending=[True, False]).reset_index(drop=True)

# Salvar CSV para o Dashboard
df_terms.to_csv(os.path.join(OUTPUT_DIR, "df_terms.csv"), index=False)
print("OK -> outputs/df_terms.csv")

# =========================================
# 3) Gerar gráficos Top termos por produto (PNG)
# =========================================
for product in sorted(df_terms["Product"].unique()):
    df_p = df_terms[df_terms["Product"] == product].head(TOP_N_TERMS).copy()
    df_p = df_p.sort_values("Score", ascending=True)  # pra plot horizontal bonito

    plt.figure(figsize=(10, 6))
    plt.barh(df_p["Term"], df_p["Score"])
    plt.title(f"Top {TOP_N_TERMS} termos (negativos) — {product}")
    plt.xlabel("TF-IDF médio")
    plt.tight_layout()

    fname = f"top_terms_{product.replace(' ', '_')}.png"
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150)
    plt.close()

    print(f"OK -> outputs/{fname}")

# =========================================
# 4) WordCloud por produto (opcional)
# =========================================
try:
    from wordcloud import WordCloud

    for product in sorted(df_neg["Product Consolidated"].dropna().unique()):
        df_temp = df_neg[df_neg["Product Consolidated"] == product]
        text_blob = " ".join(df_temp[TEXT_COL].astype(str).tolist())

        wc = WordCloud(
            width=1200,
            height=700,
            background_color="white",
            collocations=False
        ).generate(text_blob)

        plt.figure(figsize=(12, 7))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"WordCloud (negativos) — {product}")
        plt.tight_layout()

        fname = f"wordcloud_{product.replace(' ', '_')}.png"
        plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150)
        plt.close()

        print(f"OK -> outputs/{fname}")

except ImportError:
    print("WordCloud não está instalado. Se quiser gerar wordclouds:")
    print("pip install wordcloud")

OK -> outputs/df_terms.csv
OK -> outputs/top_terms_credit_cards.png
OK -> outputs/top_terms_credit_reporting.png
OK -> outputs/top_terms_debt_collection.png
OK -> outputs/top_terms_mortgages_and_loans.png
OK -> outputs/top_terms_retail_banking.png
OK -> outputs/wordcloud_credit_cards.png
OK -> outputs/wordcloud_credit_reporting.png
OK -> outputs/wordcloud_debt_collection.png
OK -> outputs/wordcloud_mortgages_and_loans.png
OK -> outputs/wordcloud_retail_banking.png
